# Part 2 — continue epoch 4 on your current L4

**Only two cells to run:** Section 1 (prepare), then Section 2 (train once). Use the **same existing L4 runtime** where restoration completed. No code upload, environment reinstall, or homework-code editing is needed. Do not rerun the old training or recovery cells.

This starts from the recovered **epoch 4 weights**, not pretrained initialization. The old optimizer/scheduler state was not saved: a new AdamW optimizer and linear scheduler are explicitly started. Batch size stays 8, learning rate stays 3e-4, weight decay stays 0, seed stays 42, and the restarted schedule has one warmup epoch. Training runs from epoch **5 through at most 20**, with early stopping after four consecutive epochs without strictly improved dev Record F1. Selection includes epoch 4 as a protected baseline.

The homework SQL execution helper is replaced **only inside the operational child process**. SQL runs read-only in an isolated, killable worker with a 120-second batch budget, preserving the assignment metric formulas and query order. The worker is terminated at the deadline so lingering SQL threads cannot keep the training process alive. Your homework source files are not changed. Section 1 includes a one-second runaway-query cleanup check.

Before epoch 5 starts, saved epoch 4 SQL is re-executed under the bounded worker; this can take up to 120 seconds and does not regenerate predictions or train. Then the log prints **`EPOCH 5/20`**. Each completed epoch is backed up before dev evaluation. Best/latest archives, logs, configuration, metrics, and dev predictions are saved under `MyDrive/NYU_NLP_HW4/part2/continuation_runs/<timestamp>/`. Keep about 1 GB free in Drive. Original training/recovery backups are not overwritten.

The held-out test set remains unused. After completion, save this notebook and ask for review before creating submission files. Section 3 is read-only and safe after reconnecting.


## 1. Prepare continuation — no training

Run once. Expected ending: `READY TO CONTINUE FROM EPOCH 4`. This does not download pretrained model weights.


In [1]:
CONTINUATION_SOURCE = "\"\"\"Continue recovered epoch 4 weights; isolated SQL workers; no homework-file edits.\n\nOnly train/dev are loaded. Checkpoint archives contain weights/config/tokenizer,\nnot optimizer state; they support later inference, not exact training resumption.\n\"\"\"\n\nimport gc\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport tarfile\nimport time\nfrom datetime import datetime, timezone\nfrom types import SimpleNamespace\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + \".partial\")\n    temporary.write_text(json.dumps(value, indent=2))\n    os.replace(temporary, path)\n\n\ndef publish_checkpoint(source, destination):\n    \"\"\"Finish a new archive before replacing this run's previous archive.\"\"\"\n    source, destination = Path(source), Path(destination)\n    required = {\"config.json\", \"model.safetensors\", \"tokenizer.json\", \"checkpoint_meta.json\"}\n    assert all((source / name).is_file() for name in required)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    temporary = destination.with_name(destination.name + \".partial\")\n    with tarfile.open(temporary, \"w\") as archive:\n        archive.add(source, arcname=source.name)\n    with tarfile.open(temporary, \"r\") as archive:\n        members = {member.name: member for member in archive.getmembers()}\n        for name in required:\n            member = members[f\"{source.name}/{name}\"]\n            assert member.size == (source / name).stat().st_size\n    # A partial archive never replaces the previous completed archive.\n    os.replace(temporary, destination)\n    return {\"path\": str(destination), \"bytes\": destination.stat().st_size}\n\n\ndef copy_predictions(sql_path, records_path, destination):\n    destination.mkdir(parents=True, exist_ok=True)\n    shutil.copy2(sql_path, destination / \"dev.sql\")\n    shutil.copy2(records_path, destination / \"dev.pkl\")\n\n\ndef run(backup):\n    import torch\n    from load_data import get_dataloader\n    from t5_utils import (\n        initialize_model, initialize_optimizer_and_scheduler,\n        save_model, load_model_from_checkpoint,\n    )\n    from train_t5 import train_epoch, eval_epoch\n    from utils import set_random_seeds\n    import utils\n    from bounded_sql_runtime import install_bounded_sql\n    install_bounded_sql(utils)\n\n    backup = Path(backup).resolve()\n    settings = json.loads((backup / \"configuration.json\").read_text())\n    args = SimpleNamespace(**settings)\n    assert args.finetune and not args.use_wandb\n    assert args.max_n_epochs > args.num_warmup_epochs >= 0\n    assert args.patience_epochs > 0 and args.learning_rate > 0\n    assert torch.cuda.is_available(), \"Connect to a GPU before starting.\"\n    set_random_seeds(args.seed)\n    started = time.monotonic()\n    status = {\"state\": \"RUNNING\", \"stage\": \"initializing\", \"completed_epochs\": 0,\n              \"backup_folder\": str(backup), \"held_out_test_used\": False}\n\n    def update(stage, **extra):\n        status.update(stage=stage, updated_utc=datetime.now(timezone.utc).isoformat(), **extra)\n        write_json(backup / \"status.json\", status)\n\n    try:\n        update(\"loading_train_and_dev\")\n        # Deliberately avoid load_t5_data()/main(): no test dataset is loaded.\n        train_loader = get_dataloader(args.batch_size, \"train\")\n        dev_loader = get_dataloader(args.test_batch_size, \"dev\")\n        assert len(train_loader.dataset) == 4225 and len(dev_loader.dataset) == 466\n        model = load_model_from_checkpoint(args, best=False)\n        base_epoch = args.completed_epochs_before_restart\n        assert base_epoch == 4, \"This continuation starts from the verified epoch 4 model.\"\n        optimizer, scheduler = initialize_optimizer_and_scheduler(args, model, len(train_loader))\n        checkpoint_dir = Path(\"checkpoints/ft_experiments\") / args.experiment_name\n        Path(\"results\").mkdir(exist_ok=True)\n        Path(\"records\").mkdir(exist_ok=True)\n        sql_path, records_path = \"results/current_dev.sql\", \"records/current_dev.pkl\"\n        gt_sql, gt_records = \"data/dev.sql\", \"records/ground_truth_dev.pkl\"\n        assert Path(gt_records).is_file()\n        metadata = {\n            \"settings\": settings, \"gpu\": torch.cuda.get_device_name(0),\n            \"parameters\": sum(p.numel() for p in model.parameters()),\n            \"trainable_parameters\": sum(p.numel() for p in model.parameters() if p.requires_grad),\n            \"train_examples\": len(train_loader.dataset), \"dev_examples\": len(dev_loader.dataset),\n            \"train_batches_per_epoch\": len(train_loader),\n            \"maximum_optimizer_steps\": len(train_loader) * args.max_n_epochs,\n            \"starting_checkpoint_epoch\": base_epoch, \"optimizer_restarted\": True,\n            \"sql_execution\": \"read-only isolated subprocess; 120-second batch budget; original metric formulas\",\n            \"warmup_steps\": len(train_loader) * args.num_warmup_epochs,\n            \"precision\": \"float32\", \"gradient_accumulation_steps\": 1,\n            \"gradient_clipping\": None, \"checkpoint_selection\": \"highest dev record F1; first on ties\",\n            \"pretrained_checkpoint\": \"google-t5/t5-small\",\n            \"checkpoint_contents\": \"weights/config/tokenizer, NOT optimizer/scheduler/RNG state\",\n            \"versions\": {name: importlib.metadata.version(name) for name in\n                         [\"torch\", \"transformers\", \"tokenizers\", \"numpy\"]},\n        }\n        write_json(backup / \"run_metadata.json\", metadata)\n        print(json.dumps(metadata, indent=2), flush=True)\n        print(\"Rechecking saved epoch 4 SQL with bounded workers (at most 120s; no generation/training).\", flush=True)\n        baseline_queries = Path(args.baseline_dev_sql).read_text().splitlines()\n        assert len(baseline_queries) == 466\n        utils.save_queries_and_records(baseline_queries, \"results/baseline_dev.sql\", \"records/baseline_dev.pkl\")\n        sql_em, record_em, f1, errors = utils.compute_metrics(\n            gt_sql, \"results/baseline_dev.sql\", gt_records, \"records/baseline_dev.pkl\",\n        )\n        best_row = {\"epoch\": base_epoch, \"record_f1\": float(f1), \"record_em\": float(record_em),\n                    \"sql_em\": float(sql_em), \"sql_error_rate\": sum(bool(e) for e in errors)/len(errors),\n                    \"dev_loss\": args.baseline_dev_loss, \"recovered_baseline\": True,\n                    \"previous_recovery_f1\": args.baseline_recovery_f1}\n        write_json(backup / \"baseline_metrics.json\", best_row)\n        write_json(backup / \"best_metrics.json\", best_row)\n        write_json(checkpoint_dir / \"best/checkpoint_meta.json\", {\n            **best_row, \"configuration\": settings, \"dev_evaluation_completed\": True,\n        })\n        status[\"best_checkpoint\"] = publish_checkpoint(checkpoint_dir / \"best\", backup / \"checkpoints/best.tar\")\n        copy_predictions(\"results/baseline_dev.sql\", \"records/baseline_dev.pkl\", backup / \"predictions/best\")\n        print(f\"Starting from epoch {base_epoch}: baseline dev Record F1={100*f1:.2f}%; weights backed up.\", flush=True)\n        print(\"CONTINUATION TRAINING STARTING. Optimizer/scheduler restarted; no held-out test use.\", flush=True)\n        history, best_f1, best_epoch, stale = [], float(f1), base_epoch, 0\n        update(\"baseline_saved_ready_to_train\", completed_epochs=base_epoch, best_epoch=best_epoch,\n               best_dev_record_f1=best_f1)\n        for epoch in range(base_epoch + 1, base_epoch + args.max_n_epochs + 1):\n            epoch_start = time.monotonic()\n            update(\"training\", current_epoch=epoch)\n            print(f\"\\nEPOCH {epoch}/{base_epoch + args.max_n_epochs}: training {len(train_loader)} batches\", flush=True)\n            train_loss = float(train_epoch(args, model, train_loader, optimizer, scheduler))\n            if not math.isfinite(train_loss):\n                raise ValueError(\"Non-finite training loss; stop and inspect.\")\n\n            # Protect the completed epoch BEFORE potentially slower dev generation.\n            update(\"backing_up_last_checkpoint\", current_epoch=epoch)\n            save_model(str(checkpoint_dir), model, best=False)\n            write_json(checkpoint_dir / \"last/checkpoint_meta.json\", {\n                \"epoch\": epoch, \"train_loss\": train_loss, \"dev_evaluation_completed\": False,\n                \"configuration\": settings,\n            })\n            last_info = publish_checkpoint(checkpoint_dir / \"last\", backup / \"checkpoints/last.tar\")\n            update(\"evaluating_dev\", completed_epochs=epoch, last_checkpoint=last_info)\n            print(f\"EPOCH {epoch}: TRAINED WEIGHTS BACKED UP TO DRIVE before dev evaluation.\", flush=True)\n\n            values = eval_epoch(args, model, dev_loader, gt_sql, sql_path, gt_records, records_path)\n            loss, f1, record_em, sql_em, error_rate = map(float, values)\n            if not all(math.isfinite(v) for v in [loss, f1, record_em, sql_em, error_rate]):\n                raise ValueError(\"Non-finite dev metrics; stop and inspect.\")\n            row = dict(epoch=epoch, train_loss=train_loss, dev_loss=loss,\n                       record_f1=f1, record_em=record_em, sql_em=sql_em, sql_error_rate=error_rate)\n            copy_predictions(sql_path, records_path, backup / \"predictions/last\")\n            improved = f1 > best_f1\n            if improved:\n                best_f1, best_epoch, stale = f1, epoch, 0\n                best_row = row\n                update(\"backing_up_best_checkpoint\")\n                save_model(str(checkpoint_dir), model, best=True)\n                write_json(checkpoint_dir / \"best/checkpoint_meta.json\", {\n                    **row, \"dev_evaluation_completed\": True, \"configuration\": settings,\n                })\n                best_info = publish_checkpoint(checkpoint_dir / \"best\", backup / \"checkpoints/best.tar\")\n                copy_predictions(sql_path, records_path, backup / \"predictions/best\")\n                status[\"best_checkpoint\"] = best_info\n            else:\n                stale += 1\n            row.update(seconds=time.monotonic() - epoch_start, improved=improved)\n            history.append(row)\n            write_json(backup / \"history.json\", history)\n            write_json(backup / \"best_metrics.json\", best_row)\n            update(\"epoch_completed\", best_epoch=best_epoch, best_dev_record_f1=best_f1,\n                   epochs_without_improvement=stale)\n            print(f\"EPOCH {epoch} COMPLETE: train loss={train_loss:.4f}, dev loss={loss:.4f}, \"\n                  f\"Record F1={100*f1:.2f}%, Record EM={100*record_em:.2f}%, \"\n                  f\"SQL EM={100*sql_em:.2f}%, SQL errors={100*error_rate:.2f}%\", flush=True)\n            print(f\"Epoch wall time including dev/backups: {row['seconds']/60:.1f} minutes. \"\n                  f\"Best epoch={best_epoch}; saved to Drive.\", flush=True)\n            if stale >= args.patience_epochs:\n                print(f\"Early stopping: {stale} epochs without improved dev Record F1.\", flush=True)\n                break\n\n        update(\"reloading_best_for_final_dev_check\")\n        del optimizer, scheduler, model\n        gc.collect()\n        torch.cuda.empty_cache()\n        best_model = load_model_from_checkpoint(args, best=True)\n        values = eval_epoch(args, best_model, dev_loader, gt_sql, \"results/best_dev.sql\",\n                            gt_records, \"records/best_dev.pkl\")\n        loss, f1, record_em, sql_em, error_rate = map(float, values)\n        assert math.isclose(f1, best_f1, rel_tol=1e-6, abs_tol=1e-8), \"Reloaded best dev F1 differs.\"\n        copy_predictions(\"results/best_dev.sql\", \"records/best_dev.pkl\", backup / \"predictions/best\")\n        final = {\"state\": \"COMPLETED\", \"experiment_name\": args.experiment_name,\n                 \"epochs_run_after_restart\": len(history), \"best_epoch\": best_epoch,\n                 \"started_from_epoch\": base_epoch, \"total_training_epochs\": history[-1][\"epoch\"],\n                 \"optimizer_restarted\": True, \"exact_resume\": False,\n                 \"dev_loss\": loss, \"record_f1\": f1, \"record_em\": record_em,\n                 \"sql_em\": sql_em, \"sql_error_rate\": error_rate,\n                 \"seconds\": time.monotonic() - started, \"held_out_test_used\": False,\n                 \"best_checkpoint_archive\": str(backup / \"checkpoints/best.tar\"),\n                 \"backup_folder\": str(backup)}\n        write_json(backup / \"final_report.json\", final)\n        update(\"completed\", state=\"COMPLETED\", final_report=final)\n        print(\"\\nPART 2 CONTINUATION AND DEV BACKUP COMPLETE\", flush=True)\n        print(json.dumps(final, indent=2), flush=True)\n        print(\"Save this notebook and ask for review. Do not submit diagnostic outputs or generate test yet.\")\n    except BaseException as error:\n        import traceback\n        update(\"failed_or_interrupted\", state=\"FAILED_OR_INTERRUPTED\", error=traceback.format_exc())\n        print(\"Run did not complete. Existing completed Drive archives/logs remain available:\", backup, flush=True)\n        raise\n\n\nif __name__ == \"__main__\":\n    run(sys.argv[1])\n"
BOUNDED_SQL_SOURCE = "\"\"\"Runtime-only replacement for compute_records: same records/metrics, killable SQL worker.\"\"\"\nimport os\nfrom pathlib import Path\nimport pickle\nimport subprocess\nimport sys\nimport tempfile\nimport time\n\nWORKER = r'''\nimport concurrent.futures, pickle, sqlite3, sys, time\nfrom pathlib import Path\nqueries_path, results_path, database, budget = sys.argv[1:]\ndeadline = time.monotonic() + float(budget)\nwith open(queries_path, \"rb\") as handle:\n    queries = pickle.load(handle)\ndef execute(index, query):\n    connection = sqlite3.connect(Path(database).resolve().as_uri() + \"?mode=ro\", uri=True)\n    connection.set_progress_handler(lambda: int(time.monotonic() >= deadline), 10000)\n    try:\n        if time.monotonic() >= deadline:\n            return index, [], \"Query timed out\"\n        rows = connection.execute(query).fetchall()\n        return index, rows, \"\"\n    except Exception as error:\n        if time.monotonic() >= deadline:\n            return index, [], \"Query timed out\"\n        return index, [], f\"{type(error).__name__}: {error}\"\n    finally:\n        connection.close()\nwith open(results_path, \"wb\") as output:\n    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as pool:\n        futures = [pool.submit(execute, i, query) for i, query in enumerate(queries)]\n        for future in concurrent.futures.as_completed(futures):\n            pickle.dump(future.result(), output, protocol=pickle.HIGHEST_PROTOCOL)\n            output.flush()\n'''\n\n\ndef compute_records_bounded(queries, database, timeout_secs=120):\n    database = Path(database).resolve()\n    if not database.is_file():\n        raise FileNotFoundError(database)\n    records, errors = [[] for _ in queries], [\"Query timed out\" for _ in queries]\n    directory = \"/content\" if Path(\"/content\").is_dir() else None\n    with tempfile.TemporaryDirectory(prefix=\"hw4_sql_worker_\", dir=directory) as tmp:\n        tmp = Path(tmp)\n        script, incoming, outgoing = tmp / \"worker.py\", tmp / \"queries.pkl\", tmp / \"results.pkl\"\n        script.write_text(WORKER)\n        with incoming.open(\"wb\") as handle:\n            pickle.dump(list(queries), handle, protocol=pickle.HIGHEST_PROTOCOL)\n        print(f\"Executing {len(queries)} SQL queries: {timeout_secs}s batch limit, isolated worker.\", flush=True)\n        started = time.monotonic()\n        process = subprocess.Popen([sys.executable, str(script), str(incoming), str(outgoing), str(database), str(timeout_secs)],\n                                   stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)\n        timed_out = False\n        try:\n            while True:\n                remaining = timeout_secs - (time.monotonic() - started)\n                if remaining <= 0:\n                    timed_out = True\n                    break\n                try:\n                    process.wait(timeout=min(15, remaining))\n                    break\n                except subprocess.TimeoutExpired:\n                    print(f\"SQL worker: {time.monotonic()-started:.0f}s elapsed; still within {timeout_secs}s limit.\", flush=True)\n            if timed_out and process.poll() is None:\n                process.terminate()\n                try:\n                    process.wait(timeout=5)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait(timeout=5)\n            stderr = process.stderr.read()\n            if not timed_out and process.returncode:\n                raise RuntimeError(f\"SQL worker failed: {stderr[-2000:]}\")\n            completed = 0\n            if outgoing.is_file():\n                with outgoing.open(\"rb\") as handle:\n                    while True:\n                        try:\n                            index, rows, error = pickle.load(handle)\n                        except (EOFError, pickle.UnpicklingError):\n                            break  # A killed worker may leave one incomplete trailing record.\n                        assert 0 <= index < len(queries)\n                        records[index], errors[index] = rows, error\n                        completed += 1\n            print(f\"SQL execution finished: {completed}/{len(queries)} completed; \"\n                  f\"{len(queries)-completed} timed out. Worker exited.\", flush=True)\n        finally:\n            if process.poll() is None:\n                process.kill()\n                process.wait(timeout=5)\n            process.stderr.close()\n    return records, errors\n\n\ndef install_bounded_sql(utils_module):\n    # Imported metric/save helpers resolve compute_records through this module's globals.\n    # This changes only this child process, never part2/utils.py on disk.\n    def bounded(queries):\n        return compute_records_bounded(queries, utils_module.DB_PATH, 120)\n    utils_module.compute_records = bounded\n"

# Notebook embeds CONTINUATION_SOURCE and BOUNDED_SQL_SOURCE before this code.
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
import importlib.util
import json
import shutil
import subprocess
import sys

assert sys.platform == "linux" and Path("/content").is_dir(), "Use your existing L4 Colab runtime."
drive.mount("/content/drive")
RECOVERY = Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/recovery_runs/20261110T132529577782Z")
report = json.loads((RECOVERY / "recovery_report.json").read_text())
context = json.loads((RECOVERY / "recovery_context.json").read_text())
assert report["state"] == "COMPLETED" and report["latest_training_epoch"] == 4
recovered_work = Path(context["work_directory"])
original = Path(context["original_run"])
PYTHON2 = "/content/hw4-part2-env/bin/python"
assert Path(PYTHON2).is_file() and recovered_work.is_dir(), (
    "The recovered local environment is gone. Stop here; do not start from pretrained weights."
)
subprocess.run([PYTHON2, "-c", "import torch; assert torch.cuda.is_available(); print('GPU:',torch.cuda.get_device_name(0))"], check=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
T5_WORK = Path("/content") / f"hw4_t5_continue_{stamp}"
T5_BACKUP = original.parent.parent / "continuation_runs" / stamp
T5_WORK.mkdir(exist_ok=False)
T5_BACKUP.mkdir(parents=True, exist_ok=False)

spec = importlib.util.spec_from_file_location("continuation_restore_tools", recovered_work / "recovery_tools.py")
tools = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tools)
tools.restore_source(original, T5_WORK)
settings = dict(context["configuration"])
settings.update(
    max_n_epochs=16, completed_epochs_before_restart=4,
    baseline_dev_sql=report["checkpoints"]["last"]["sql_path"],
    baseline_dev_loss=report["checkpoints"]["last"]["dev_loss"],
    baseline_recovery_f1=report["checkpoints"]["last"]["record_f1"],
    optimizer_restarted=True, exact_resume=False,
    recovery_report=str(RECOVERY / "recovery_report.json"),
)
source_checkpoint = Path(context["checkpoints"]["last"]["local_directory"])
assert json.loads((source_checkpoint / "checkpoint_meta.json").read_text())["epoch"] == 4
checkpoint_root = T5_WORK / "checkpoints/ft_experiments" / settings["experiment_name"]
for name in ["best", "last"]:
    shutil.copytree(source_checkpoint, checkpoint_root / name)
    assert tools.sha256_file(checkpoint_root / name / "model.safetensors") == tools.sha256_file(source_checkpoint / "model.safetensors")
shutil.copytree(original / "source", T5_BACKUP / "source")
shutil.copy2(original / "source_sha256.json", T5_BACKUP / "source_sha256.json")
for name in ["environment.json", "pip_freeze.txt"]:
    if (RECOVERY / name).is_file():
        shutil.copy2(RECOVERY / name, T5_BACKUP / name)
(T5_BACKUP / "configuration.json").write_text(json.dumps(settings, indent=2))
(T5_BACKUP / "recovered_baseline_report.json").write_text(json.dumps(report, indent=2))
T5_RUNNER = T5_WORK / "continue_training.py"
T5_RUNNER.write_text(CONTINUATION_SOURCE)
sql_runtime = T5_WORK / "bounded_sql_runtime.py"
sql_runtime.write_text(BOUNDED_SQL_SOURCE)
for path in [T5_RUNNER, sql_runtime]:
    shutil.copy2(path, T5_BACKUP / path.name)

# A 1-second runaway-query check verifies cleanup without training or modifying the DB.
spec = importlib.util.spec_from_file_location("sql_cleanup_smoke", sql_runtime)
sql_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sql_module)
rows, errors = sql_module.compute_records_bounded([
    "SELECT 1",
    "WITH RECURSIVE t(x) AS (SELECT 1 UNION ALL SELECT x+1 FROM t) SELECT sum(x) FROM t",
], T5_WORK / "data/flight_database.db", 1)
assert rows[0] == [(1,)] and not errors[0] and errors[1] == "Query timed out"
(T5_BACKUP / "status.json").write_text(json.dumps({
    "state": "PREPARED_NOT_STARTED", "starting_epoch": 4,
    "next_epoch": 5, "backup_folder": str(T5_BACKUP),
    "optimizer_restarted": True, "exact_resume": False,
}, indent=2))
print("SQL TIMEOUT CLEANUP CHECK PASSED")
print("READY TO CONTINUE FROM EPOCH 4. Training has NOT started.")
print("Original/recovery Drive backups unchanged.")
print("NEW CONTINUATION BACKUP:", T5_BACKUP)
print("Run Section 2 ONCE. It rechecks saved SQL (up to 120 seconds), then starts EPOCH 5/20.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Executing 2 SQL queries: 1s batch limit, isolated worker.
SQL worker: 1s elapsed; still within 1s limit.
SQL execution finished: 1/2 completed; 1 timed out. Worker exited.
SQL TIMEOUT CLEANUP CHECK PASSED
READY TO CONTINUE FROM EPOCH 4. Training has NOT started.
Original/recovery Drive backups unchanged.
NEW CONTINUATION BACKUP: /content/drive/MyDrive/NYU_NLP_HW4/part2/continuation_runs/20261110T140531341373Z
Run Section 2 ONCE. It rechecks saved SQL (up to 120 seconds), then starts EPOCH 5/20.


## 2. Continue training — run ONCE

This starts real training from epoch 5 after a bounded SQL baseline check (up to 120 seconds). Leave it running. Every completed training epoch is backed up before dev evaluation.


In [2]:
# Run once. This is the long-running cell that actually trains the model.
import json
import subprocess
import os
import signal
from datetime import datetime, timezone

assert "T5_WORK" in globals() and T5_RUNNER.is_file(), "Run preparation first."
assert json.loads((T5_BACKUP / "status.json").read_text())["state"] == "PREPARED_NOT_STARTED", (
    "This run was already started. Do not accidentally train twice."
)
with (T5_WORK / ".training_started").open("x") as marker:
    marker.write(datetime.now(timezone.utc).isoformat())

print("STARTING CONTINUATION FROM EPOCH 4. Drive backup:", T5_BACKUP, flush=True)
print("The recovered baseline is protected; first recheck saved SQL, then begin epoch 5.", flush=True)
process = None
try:
    with (T5_BACKUP / "training.log").open("w", buffering=1) as log:
        process = subprocess.Popen(
            [PYTHON2, "-u", str(T5_RUNNER), str(T5_BACKUP)],
            cwd=T5_WORK, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
        if return_code:
            raise RuntimeError(f"Training process exited with code {return_code}. Inspect training.log.")
except BaseException:
    if process is not None and process.poll() is None:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=15)
    status_path = T5_BACKUP / "status.json"
    status = json.loads(status_path.read_text()) if status_path.is_file() else {}
    status.update(state="FAILED_OR_INTERRUPTED", updated_utc=datetime.now(timezone.utc).isoformat())
    status_path.write_text(json.dumps(status, indent=2))
    print("Stopped or failed. Completed Drive checkpoints/logs remain at:", T5_BACKUP)
    raise
else:
    assert json.loads((T5_BACKUP / "status.json").read_text())["state"] == "COMPLETED"
    print("\nTRAINING COMPLETE. Everything needed for review is in:", T5_BACKUP)
    print("Save this notebook. Do not rerun training; ask for review of the dev results.")


STARTING CONTINUATION FROM EPOCH 4. Drive backup: /content/drive/MyDrive/NYU_NLP_HW4/part2/continuation_runs/20261110T140531341373Z
The recovered baseline is protected; first recheck saved SQL, then begin epoch 5.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/content/hw4-part2-env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
{
  "settings": {
    "finetune": true,
    "experiment_name": "experiment",
    "seed": 42,
    "optimizer_type": "AdamW",
    "learning_rate": 0.0003,
    "weight_decay": 0.0,
    "scheduler_type": "linear",
    "num_warmup_epochs": 1,
    "max_n_epochs": 16,
    "patience_epochs": 4,
    "batch_size": 8,
    "test_batch_size": 8,
    "use_wandb": false,
    "max_new_tokens":

## 3. Inspect saved progress — read-only

Safe after completion or reconnecting. Do not rerun preparation/training merely to inspect results.


In [3]:
# Read-only: safe to run after reconnecting, without re-running training.
from pathlib import Path
from google.colab import drive
import json

drive.mount("/content/drive")
if "T5_BACKUP" not in globals():
    runs = sorted(Path("/content/drive/MyDrive/NYU_NLP_HW4/part2/continuation_runs").glob("*"))
    assert runs, "No saved Part 2 continuation runs found."
    T5_BACKUP = runs[-1]
print("Inspecting:", T5_BACKUP)
for name in ["status.json", "best_metrics.json", "final_report.json"]:
    path = T5_BACKUP / name
    if path.is_file():
        print("\n" + name + ":\n" + path.read_text())
for name in ["best.tar", "last.tar"]:
    path = T5_BACKUP / "checkpoints" / name
    print(name, f"{path.stat().st_size / 1024**2:.1f} MiB" if path.is_file() else "not yet saved")
log = T5_BACKUP / "training.log"
if log.is_file():
    print("\nLast log lines:\n" + "\n".join(log.read_text(errors="replace").splitlines()[-15:]))
print("A stale RUNNING status after runtime loss does not prove the process is still alive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Inspecting: /content/drive/MyDrive/NYU_NLP_HW4/part2/continuation_runs/20261110T140531341373Z

status.json:
{
  "state": "COMPLETED",
  "stage": "completed",
  "completed_epochs": 20,
  "backup_folder": "/content/drive/MyDrive/NYU_NLP_HW4/part2/continuation_runs/20261110T140531341373Z",
  "held_out_test_used": false,
  "updated_utc": "2026-11-10T16:02:21.564980+00:00",
  "best_checkpoint": {
    "path": "/content/drive/MyDrive/NYU_NLP_HW4/part2/continuation_runs/20261110T140531341373Z/checkpoints/best.tar",
    "bytes": 245299200
  },
  "best_epoch": 17,
  "best_dev_record_f1": 0.8724690312780573,
  "current_epoch": 20,
  "last_checkpoint": {
    "path": "/content/drive/MyDrive/NYU_NLP_HW4/part2/continuation_runs/20261110T140531341373Z/checkpoints/last.tar",
    "bytes": 245299200
  },
  "epochs_without_improvement": 3,
  "final_report": {
    "state": "COMPL